# Urban Context Feature Extraction

This notebook extracts station-level urban context features for Sydney permanent traffic monitoring stations. Each station is represented by a 500 metre buffer, and OpenStreetMap features are aggregated within that buffer.


In [ ]:
from pathlib import Path
import re
import unicodedata

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio


## Paths and Configuration


In [ ]:
station_path = Path("sydney_permanent_stations_candidate_1.csv")
pbf_path = Path("data/osm/Sydney.osm.pbf")
output_dir = Path("POIs_output")
output_dir.mkdir(parents=True, exist_ok=True)

metric_crs = "EPSG:7856"
buffer_distance_m = 500


## Load Station Anchors


In [ ]:
stations = pd.read_csv(station_path)

stations_gdf = gpd.GeoDataFrame(
    stations.copy(),
    geometry=gpd.points_from_xy(
        stations["wgs84_longitude"],
        stations["wgs84_latitude"],
    ),
    crs="EPSG:4326",
)

stations_m = stations_gdf.to_crs(metric_crs)
station_buffers = stations_m[["station_key", "geometry"]].copy()
station_buffers["geometry"] = station_buffers.geometry.buffer(buffer_distance_m)

assert stations_m["station_key"].is_unique


## Read and Prepare OSM Layers

The local `.osm.pbf` extract stores many useful OSM tags in `other_tags`. The helper functions below extract the tags needed for POI, building, and land-use features.


In [ ]:
target_tags = [
    "name",
    "amenity",
    "shop",
    "public_transport",
    "leisure",
    "tourism",
    "healthcare",
    "landuse",
    "natural",
    "building",
]

def extract_osm_tag(other_tags, key):
    if pd.isna(other_tags):
        return pd.NA
    pattern = rf'"{re.escape(key)}"=>"([^"]*)"'
    match = re.search(pattern, str(other_tags))
    return match.group(1) if match else pd.NA

def prepare_osm_layer(gdf, source_layer):
    prepared = gdf.copy()
    prepared["source_layer"] = source_layer

    for tag in target_tags:
        if tag not in prepared.columns:
            prepared[tag] = prepared["other_tags"].apply(
                lambda value: extract_osm_tag(value, tag)
            )

    prepared["osm_uid"] = prepared.apply(
        lambda row: (
            f"r_{row['osm_id']}"
            if pd.notna(row.get("osm_id"))
            else f"w_{row['osm_way_id']}"
        ),
        axis=1,
    )
    return prepared

points = pyogrio.read_dataframe(pbf_path, layer="points")
polygons = pyogrio.read_dataframe(pbf_path, layer="multipolygons")

points_clean = prepare_osm_layer(points, source_layer="point")
polygons_clean = prepare_osm_layer(polygons, source_layer="multipolygon")


## POI Features


In [ ]:
poi_tags = [
    "amenity",
    "shop",
    "public_transport",
    "leisure",
    "tourism",
    "healthcare",
]

points_poi = points_clean.loc[points_clean[poi_tags].notna().any(axis=1)].copy()
polygons_poi = polygons_clean.loc[polygons_clean[poi_tags].notna().any(axis=1)].copy()

polygons_poi_points = polygons_poi.copy()
polygons_poi_points["geometry"] = polygons_poi_points.geometry.representative_point()

poi_columns = [
    "osm_uid",
    "name",
    "amenity",
    "shop",
    "public_transport",
    "leisure",
    "tourism",
    "healthcare",
    "geometry",
    "source_layer",
]

all_poi = gpd.GeoDataFrame(
    pd.concat(
        [points_poi[poi_columns], polygons_poi_points[poi_columns]],
        ignore_index=True,
    ),
    geometry="geometry",
    crs=points_poi.crs,
)


### Remove Likely Duplicate POIs

Some POIs appear as both a point and a polygon. Named polygon POIs are removed when their representative point falls within 20 metres of a point POI with the same normalised name.


In [ ]:
def normalize_poi_name(value):
    if pd.isna(value):
        return pd.NA
    value = unicodedata.normalize("NFKC", str(value)).lower().strip()
    value = re.sub(r"[^\w\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value if value else pd.NA

all_poi["name_clean"] = all_poi["name"].apply(normalize_poi_name)
all_poi_m = all_poi.to_crs(metric_crs)

named_points = all_poi_m.loc[
    (all_poi_m["source_layer"].eq("point"))
    & all_poi_m["name_clean"].notna()
].copy()
named_polygons = all_poi_m.loc[
    (all_poi_m["source_layer"].eq("multipolygon"))
    & all_poi_m["name_clean"].notna()
].copy()

named_points["original_index"] = named_points.index
named_polygons["original_index"] = named_polygons.index

point_buffers = named_points[["name_clean", "original_index", "geometry"]].copy()
point_buffers["geometry"] = point_buffers.geometry.buffer(20)

duplicate_matches = gpd.sjoin(
    named_polygons[["osm_uid", "name", "name_clean", "original_index", "geometry"]],
    point_buffers[["name_clean", "original_index", "geometry"]],
    how="inner",
    predicate="within",
    lsuffix="polygon",
    rsuffix="point",
)

duplicate_matches = duplicate_matches.loc[
    duplicate_matches["name_clean_polygon"].eq(
        duplicate_matches["name_clean_point"]
    )
]

duplicate_polygon_indices = (
    duplicate_matches["original_index_polygon"].dropna().unique()
)
all_poi_dedup_m = all_poi_m.drop(index=duplicate_polygon_indices).copy()


### Aggregate POI Categories to Station Buffers


In [ ]:
poi_station_matches = gpd.sjoin(
    all_poi_dedup_m,
    station_buffers[["station_key", "geometry"]],
    how="inner",
    predicate="within",
).copy()

food_values = {"restaurant", "cafe", "fast_food", "pub", "bar", "food_court", "biergarten"}
education_values = {"school", "university", "college", "kindergarten", "childcare", "language_school", "music_school"}
healthcare_amenity_values = {"hospital", "clinic", "doctors", "dentist", "pharmacy", "nursing_home"}
public_transport_values = {"platform", "station", "stop_position"}
leisure_values = {"park", "garden", "playground", "pitch", "sports_centre", "fitness_centre", "stadium"}
tourism_values = {"hotel", "motel", "hostel", "guest_house", "attraction", "museum", "artwork", "information"}

poi_station_matches["is_food"] = poi_station_matches["amenity"].isin(food_values)
poi_station_matches["is_education"] = poi_station_matches["amenity"].isin(education_values)
poi_station_matches["is_healthcare"] = (
    poi_station_matches["amenity"].isin(healthcare_amenity_values)
    | poi_station_matches["healthcare"].notna()
)
poi_station_matches["is_public_transport"] = poi_station_matches["public_transport"].isin(public_transport_values)
poi_station_matches["is_leisure"] = poi_station_matches["leisure"].isin(leisure_values)
poi_station_matches["is_tourism"] = poi_station_matches["tourism"].isin(tourism_values)
poi_station_matches["is_shop"] = poi_station_matches["shop"].notna()

category_columns = [
    "is_food",
    "is_education",
    "is_healthcare",
    "is_public_transport",
    "is_leisure",
    "is_tourism",
    "is_shop",
]

station_poi_counts = (
    poi_station_matches.groupby("station_key")[category_columns]
    .sum()
    .reset_index()
    .rename(
        columns={
            "is_food": "poi_food_count_500m",
            "is_education": "poi_education_count_500m",
            "is_healthcare": "poi_healthcare_count_500m",
            "is_public_transport": "poi_public_transport_count_500m",
            "is_leisure": "poi_leisure_count_500m",
            "is_tourism": "poi_tourism_count_500m",
            "is_shop": "poi_shop_count_500m",
        }
    )
)

poi_count_columns = [
    "poi_food_count_500m",
    "poi_education_count_500m",
    "poi_healthcare_count_500m",
    "poi_public_transport_count_500m",
    "poi_leisure_count_500m",
    "poi_tourism_count_500m",
    "poi_shop_count_500m",
]

station_base_columns = [
    "station_key",
    "station_id",
    "name",
    "road_name",
    "intersection",
    "lga",
    "suburb",
    "post_code",
    "wgs84_latitude",
    "wgs84_longitude",
]

station_poi_features = (
    stations[station_base_columns]
    .drop_duplicates("station_key")
    .merge(station_poi_counts, on="station_key", how="left")
)
station_poi_features[poi_count_columns] = (
    station_poi_features[poi_count_columns].fillna(0).astype(int)
)
station_poi_features["poi_data_missing"] = station_poi_features[poi_count_columns].sum(axis=1).eq(0)
station_poi_features.to_csv(output_dir / "station_poi_features_500m.csv", index=False)


## Building Features


In [ ]:
buildings = polygons_clean.loc[polygons_clean["building"].notna()].copy()
buildings_m = buildings.to_crs(metric_crs)

building_station_matches = gpd.sjoin(
    buildings_m,
    station_buffers[["station_key", "geometry"]],
    how="inner",
    predicate="intersects",
).copy()

building_station_matches["building_uid"] = building_station_matches.apply(
    lambda row: (
        f"r_{row['osm_id']}"
        if pd.notna(row.get("osm_id"))
        else f"w_{row['osm_way_id']}"
    ),
    axis=1,
)

station_building_counts = (
    building_station_matches.groupby("station_key")["building_uid"]
    .nunique()
    .rename("building_count_500m")
    .reset_index()
)

station_building_features = (
    stations_m[["station_key"]]
    .drop_duplicates("station_key")
    .merge(station_building_counts, on="station_key", how="left")
)
station_building_features["building_count_500m"] = (
    station_building_features["building_count_500m"].fillna(0).astype(int)
)

buffer_area_km2 = np.pi * (0.5 ** 2)
station_building_features["building_density_per_km2_500m"] = (
    station_building_features["building_count_500m"] / buffer_area_km2
)
station_building_features["building_data_missing"] = station_building_features["building_count_500m"].eq(0)
station_building_features.to_csv(output_dir / "station_building_features_500m.csv", index=False)


## Land-Use Features


In [ ]:
landuse_category_map = {
    "residential": "residential",
    "commercial": "commercial",
    "retail": "commercial",
    "mixed": "commercial",
    "industrial": "industrial",
    "depot": "industrial",
    "quarry": "industrial",
    "landfill": "industrial",
    "yard": "industrial",
    "pipeline": "industrial",
    "education": "institutional",
    "schoolyard": "institutional",
    "religious": "institutional",
    "churchyard": "institutional",
    "civic": "institutional",
    "institutional": "institutional",
    "community": "institutional",
    "public": "institutional",
    "observatory": "institutional",
    "military": "institutional",
    "grass": "green_recreation",
    "recreation_ground": "green_recreation",
    "village_green": "green_recreation",
    "forest": "green_recreation",
    "meadow": "green_recreation",
    "orchard": "green_recreation",
    "farmyard": "green_recreation",
    "farmland": "green_recreation",
    "railway": "transport_other",
    "highway": "transport_other",
    "garages": "transport_other",
    "cemetery": "transport_other",
    "construction": "transport_other",
    "brownfield": "transport_other",
}

landuse = polygons_clean.loc[polygons_clean["landuse"].notna()].copy()
landuse["landuse_type"] = landuse["landuse"].astype("string").str.strip().str.lower()
landuse["landuse_category"] = (
    landuse["landuse_type"].map(landuse_category_map).fillna("transport_other")
)

landuse_m = landuse.to_crs(metric_crs)
landuse_m["geometry"] = landuse_m.geometry.make_valid()

landuse_station_matches = gpd.sjoin(
    landuse_m[["osm_id", "osm_way_id", "landuse_type", "landuse_category", "geometry"]],
    station_buffers[["station_key", "geometry"]],
    how="inner",
    predicate="intersects",
).copy()

landuse_intersections = landuse_station_matches.merge(
    station_buffers[["station_key", "geometry"]].rename(columns={"geometry": "buffer_geometry"}),
    on="station_key",
    how="left",
)
landuse_intersections["intersection_geometry"] = landuse_intersections.geometry.intersection(
    landuse_intersections["buffer_geometry"]
)
landuse_intersections["intersection_area_m2"] = landuse_intersections["intersection_geometry"].area

station_landuse_area = (
    landuse_intersections.groupby(["station_key", "landuse_category"], as_index=False)["intersection_area_m2"]
    .sum()
)
station_total_landuse_area = (
    station_landuse_area.groupby("station_key", as_index=False)["intersection_area_m2"]
    .sum()
    .rename(columns={"intersection_area_m2": "total_mapped_landuse_area_m2"})
)
station_landuse_area_normalized = station_landuse_area.merge(
    station_total_landuse_area, on="station_key", how="left"
)
station_landuse_area_normalized["landuse_pct_500m"] = (
    station_landuse_area_normalized["intersection_area_m2"]
    / station_landuse_area_normalized["total_mapped_landuse_area_m2"]
    * 100
)

station_landuse_features = (
    station_landuse_area_normalized.pivot(
        index="station_key", columns="landuse_category", values="landuse_pct_500m"
    )
    .fillna(0)
    .reset_index()
)
station_landuse_features.columns.name = None
station_landuse_features = station_landuse_features.rename(
    columns={
        "commercial": "commercial_pct_500m",
        "green_recreation": "green_recreation_pct_500m",
        "industrial": "industrial_pct_500m",
        "institutional": "institutional_pct_500m",
        "residential": "residential_pct_500m",
        "transport_other": "transport_other_pct_500m",
    }
)

landuse_feature_columns = [
    "commercial_pct_500m",
    "green_recreation_pct_500m",
    "industrial_pct_500m",
    "institutional_pct_500m",
    "residential_pct_500m",
    "transport_other_pct_500m",
]

station_landuse_features_final = (
    stations[station_base_columns]
    .drop_duplicates("station_key")
    .merge(station_landuse_features, on="station_key", how="left")
)
station_landuse_features_final[landuse_feature_columns] = (
    station_landuse_features_final[landuse_feature_columns].fillna(0)
)
station_landuse_features_final["landuse_data_missing"] = (
    station_landuse_features_final[landuse_feature_columns].sum(axis=1).eq(0)
)
station_landuse_features_final.to_csv(output_dir / "station_landuse_features_500m.csv", index=False)


## Merge Final Outputs


In [ ]:
stations_final = stations_gdf.copy()
poi_features = pd.read_csv(output_dir / "station_poi_features_500m.csv")
building_features = pd.read_csv(output_dir / "station_building_features_500m.csv")
landuse_features = pd.read_csv(output_dir / "station_landuse_features_500m.csv")

for dataframe in [stations_final, poi_features, building_features, landuse_features]:
    dataframe["station_key"] = dataframe["station_key"].astype("string").str.strip()

poi_feature_columns = ["station_key", *poi_count_columns, "poi_data_missing"]
building_feature_columns = [
    "station_key",
    "building_count_500m",
    "building_density_per_km2_500m",
    "building_data_missing",
]
landuse_export_columns = ["station_key", *landuse_feature_columns, "landuse_data_missing"]

station_urban_context = (
    stations_final.merge(poi_features[poi_feature_columns], on="station_key", how="left")
    .merge(building_features[building_feature_columns], on="station_key", how="left")
    .merge(landuse_features[landuse_export_columns], on="station_key", how="left")
)

final_gpkg_path = output_dir / "sydney_station_urban_context_500m.gpkg"
final_csv_path = output_dir / "sydney_station_urban_context_500m.csv"

station_urban_context.to_file(final_gpkg_path, layer="urban_context_500m", driver="GPKG")
station_urban_context.drop(columns="geometry").to_csv(final_csv_path, index=False)

assert len(station_urban_context) == len(stations_final)
station_urban_context.shape
